In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
import panel as pn

from tqdm import tqdm
from scipy import stats
from pathlib import Path
from pprint import pprint
from holoviews import opts
from bokeh.io import output_notebook
from bokeh.palettes import Spectral10, Colorblind8
from concurrent.futures import ProcessPoolExecutor
from scipy.ndimage import gaussian_filter1d



font_dict = {'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12}
hv.opts.defaults(
    hv.opts.Curve(width=600, height=400, tools=['hover'], fontsize=font_dict),
    hv.opts.Scatter(width=600, height=400, size=8, tools=['hover'], fontsize=font_dict),
    hv.opts.Histogram(width=600, height=400, fontsize=font_dict),
    hv.opts.Bars(width=600, height=400, fontsize=font_dict),
)

output_notebook()
hv.extension('bokeh')

Loading BokehJS ...

In [24]:
monkey = 'fiona' # 'yasmin'  or 'fiona' 

# Load the pickle file
save_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data'
pickle_file = save_path / f'msn_{monkey}_cell_trial_data.pkl'
cell_df = pd.read_pickle(pickle_file)
print(f"Unified DataFrame loaded from: {pickle_file}")
print(f"DataFrame shape: {cell_df.shape}")
print(cell_df.columns)
cell_df.head()

Unified DataFrame loaded from: /home/barak/Projects/population_analysis/data/unified_cell_trial_data/msn_fiona_cell_trial_data.pkl
DataFrame shape: (844694, 27)
Index(['cell_ID', 'cell_type', 'maestro_ID', 'problem', 'grade', 'filename',
       'trial_name', 'reaction_time', 'go_cue', 'stop_cue', 'trial_failed',
       'ssd_len', 'ssd_number', 'type', 'first_relevant_saccade',
       'segs_durations', 'segs_times', 'trial_length', 'screen_rotation',
       'saccades', 'blinks', 'dir', 'neural_data', 'session', 'plexon_session',
       'trial_number', 'trial_session'],
      dtype='object')


,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,9867,msn,2,NaN,9,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,[1978.93],fi210824,a,255,fi210824a
1,9868,msn,3,NaN,8,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,"[40.85, 1162.37, 1952.2199999999998, 2059.2999...",fi210824,a,255,fi210824a
2,9869,msn,4,NaN,8,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,[],fi210824,a,255,fi210824a
3,9870,msn,5,NaN,9,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,[],fi210824,a,255,fi210824a
4,9871,msn,6,NaN,8,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,"[206.02, 257.5, 521.7, 611.32, 773.57, 818.050...",fi210824,a,255,fi210824a


In [25]:
class DataFrameAnalyzer:
    def __init__(self, df):
        self.data = df.copy()
        self.psth_matrix = None

    @property
    def cells(self):
        return self.data['cell_ID'].unique().tolist()

    @property
    def sessions(self):
        return self.data['trial_session'].unique().tolist()

    @property
    def trials(self):
        return self.data['filename'].unique().tolist()
    
    @property
    def monkeys(self):
        return {s[:2] for s in self.sessions}

    def get_sessions_by_cell(self, cell_id):
        return self.data[self.data['cell_ID'] == cell_id]['trial_session'].unique().tolist()
    
    def has_multiple_sessions(self):
        "Is there a cell that has more than one session?"
        return any(self.data.groupby('cell_ID')['trial_session'].nunique() > 1)
    
    def get_trials_by_cell(self, cell_id):
        return self.data[self.data['cell_ID'] == cell_id]['filename'].unique().tolist()
    
    def get_cell_instance(self, cell_id):
        cell_df = self.data[self.data['cell_ID'] == cell_id]
        if cell_df.empty:
            return None
        return Cell(cell_df)
    
    def find_cells_with_8_directions(self):
        cells_with_8_directions = []
        for cell_id in self.cells:
            cell_instance = self.get_cell_instance(cell_id)
            if cell_instance and len(cell_instance.directions) == 8:
                cells_with_8_directions.append(cell_id)
        return cells_with_8_directions
    
    def get_PSTH_matrix(
        self, remove_nan_rows=False, epok_buffer=0,
        epok=[-300, 500], alignment_point='first_relevant_saccade', smooth_ker_size=25
    ):
        
        # if self.psth_matrix is not None:
        #     # todo: check concfig match
        #     return self.psth_matrix
        
        psth_lst = []
        for i, id in enumerate(self.cells):
            cell = self.get_cell_instance(id)
            # if alignment_point == 'first_relevant_saccade' and cell.data['first_relevant_saccade'].isnan().all():
            #     continue
            if remove_nan_rows and cell.directions.shape[0] != 8:
                continue
            else:
                psth_row = cell.get_PSTH_row(
                    epok=epok, alignment_point=alignment_point, 
                    smooth_ker_size=smooth_ker_size, epok_buffer=epok_buffer
                )
                psth_lst.append(psth_row)

        self.psth_matrix = np.array(psth_lst)
        return self.psth_matrix
    
    def _PSTH_matrix_factory(
        self, remove_nan_rows=False, epok_buffer=0,
        alignment_point='first_relevant_saccade', smooth_ker_size=25
    ):
        func = lambda epok: self.get_PSTH_matrix(
            remove_nan_rows=remove_nan_rows, epok_buffer=epok_buffer,
            epok=epok, alignment_point=alignment_point, smooth_ker_size=smooth_ker_size
        )
        return func
    
    def get_PSTH_matrixes_per_range(
        self, epok_len=50, step=1, epoks_range=[-50,50], 
        alignment_point='go_cue', remove_nan_rows=True,
        smooth_ker_size=25, epok_buffer=0
    ):
        epok_start_times = np.arange(epoks_range[0], epoks_range[1], step)
        epok_end_times = epok_start_times + epok_len    
        epoks = np.vstack((epok_start_times, epok_end_times)).T
        print(epoks.shape)
        # def get_mats(epok):
        #     psth_matrix = analyzer.get_PSTH_matrix(
        #         remove_nan_rows=remove_nan_rows, epok=epok, alignment_point=alignment_point, 
        #         smooth_ker_size=smooth_ker_size, epok_buffer=epok_buffer
        #     )
        #     print(epok, psth_matrix.shape)
        #     print(f"Free memory: {os.popen('free -m').readlines()[1].split()[3]} MB")
        #     return psth_matrix
        
        # get_mats = self._PSTH_matrix_factory(
        #     remove_nan_rows=remove_nan_rows, epok_buffer=epok_buffer,
        #     alignment_point=alignment_point, smooth_ker_size=smooth_ker_size
        # )

        # with ProcessPoolExecutor() as executor:
        #     mats = list(executor.map(get_mats, epoks))

        with ProcessPoolExecutor() as executor:
            futures = [
                executor.submit(
                    self.get_PSTH_matrix, epok=epok, alignment_point=alignment_point, 
                    smooth_ker_size=smooth_ker_size, epok_buffer=epok_buffer    
                ) 
                for epok in epoks
            ]
            mats = [future.result() for future in futures]

        return mats

            
    
class Cell:
    
    DIR_COLORS = {int(dir): color for dir, color in zip(np.arange(0,360,45, dtype=int), Colorblind8)}

    def __init__(self, cell_df):
        self.data = cell_df.copy().reset_index(drop=True)
        self.data = self.data.sort_values(by='dir').reset_index(drop=True)
        self.cell_id = cell_df.iloc[0]['cell_ID']
        self.trials = cell_df['filename'].unique()
        self.trial_sessions = cell_df['trial_session'].unique()
        self.trial_numbers = cell_df['trial_number'].unique()
        self.trial_ids = cell_df['filename'].unique()
        self.trial_sessions = cell_df['trial_session'].unique()
        self.trial_ids = cell_df['filename'].unique()
        assert cell_df['trial_session'].unique().shape[0] == 1
        self.trial_session = cell_df.iloc[0]['trial_session']
        self.directions = np.sort(cell_df['dir'].unique())
        self.trial_types = cell_df['type'].unique()
        self.psth_df = None

    def t_0_initer(self, row, alignment_point='motion_onset'):
        try:
            row = self.data.iloc[row]
        except ValueError:
            row = row

        if alignment_point == 'go_cue':
            t_0 = row['go_cue']
        elif alignment_point == 'first_relevant_saccade':
            t_0 = row['first_relevant_saccade'][0]
        elif alignment_point == 'stop_cue':
            t_0 = row['stop_cue']
        else:
            t_0 = alignment_point

        return t_0


    def create_alinged_times(self, alignment_point='go_cue'):

        def alignment_helper(row, alignment_point=alignment_point):
            try:
                t_0 = self.t_0_initer(row, alignment_point)
                return np.arange(-t_0, (row['trial_length'] - t_0 + 1), 1)
            except Exception as e:
                print(row)
                raise e
        
        self.data['aligned_times'] = self.data.apply(alignment_helper, axis=1)
        return self.data['aligned_times']
    
    def create_aligned_spikes_array(self, alignment_point='go_cue'):
        
        def create_spikes_in_time(row):
            spikes = np.zeros_like(row['aligned_times'])
            spike_times = np.array(row['neural_data'], dtype=int)
            spikes[spike_times] = 1
            return spikes
        
        self.create_alinged_times(alignment_point)
        self.data['spikes_in_time'] =  self.data.apply(
            create_spikes_in_time, 
            axis=1
        )
        return self.data['spikes_in_time']
        
    def fix_spike_times_to_alignment(self, alignment_point='go_cue'):
        self.data['fixed_neural_data'] = self.data.apply(
            lambda x: x['neural_data'].astype(int) if isinstance(x['neural_data'], np.ndarray) else np.array([x['neural_data']], dtype=int), 
            axis=1
        )
        self.data['fixed_neural_data'] = self.data.apply(
            lambda row: row['fixed_neural_data'] - self.t_0_initer(row, alignment_point), 
            axis=1
        )
        return self.data['fixed_neural_data']
    
    def crop_aligned_spikes_to_epok(self, epok_start=-500, epok_end=800, alignment_point='go_cue'):
        # if 'aligned_times' not in self.data.columns:
        self.create_aligned_spikes_array(alignment_point)

        def crop_spikes_to_epok(row, epok_start=epok_start, epok_end=epok_end):
            try:
                start_idx = np.argwhere(row['aligned_times'] == epok_start)[0][0]
                end_idx = np.argwhere(row['aligned_times'] == epok_end)[0][0]
                return row['spikes_in_time'][start_idx:end_idx]
            except IndexError as e:
                return None
            except Exception as e:
                print(row)
                print(epok_start, epok_end)
                print(row['aligned_times'][0], row['aligned_times'][-1])
                raise e
        
        self.data['spikes_in_epok'] = self.data.apply(crop_spikes_to_epok, axis=1)
        return self.data['spikes_in_epok']

    def plot_cell_raster(self, alignment_point='go_cue', epok=None):
        if epok is not None:
            epok_start, epok_end = epok
            try:
                self.data['spikes_in_epok']
            except KeyError as e:
                self.crop_aligned_spikes_to_epok(epok_start, epok_end, alignment_point)
                 
        def get_spikes_data(row, epok=epok):
            if epok is not None:
                    return np.argwhere(row['spikes_in_epok'] != 0).flatten() + epok[0]
            else:
                spikes = row['fixed_neural_data'].astype(int)
                if len(spikes) == 0:
                    return np.array(row[-100])  # return an array with a single invalid spike time
                return spikes
        
        
        self.fix_spike_times_to_alignment(alignment_point)
        # overlay = hv.NdOverlay(
        #     {
        #         i: hv.Spikes(
        #             get_spikes_data(self.data.iloc[i]), kdims='Time'
        #         ).opts(
        #             position=i, color=self.DIR_COLORS[self.data.iloc[i]['dir']]
        #         ) 
        #         for i in range(self.data.shape[0])
        #     }
        # ).opts(
        #     # yticks=[i for i in np.arange(self.data.shape[0], step=10)],
        #     ylabel='Trial #', show_legend=True,
        #     title=f"Cell {self.cell_id} Raster Plot aligned to {alignment_point}"
        # )

        # overlay = hv.Spikes(
        #     get_spikes_data(self.data.iloc[1]), kdims='Time'
        # ).opts(
        #     position=1, color=self.DIR_COLORS[self.data.iloc[1]['dir']]
        # )

        # return overlay.opts(
        #     opts.Spikes(spike_length=1, line_width=5, width=800, height=400),
        #     opts.NdOverlay(show_legend=False, show_grid=True)
        # )
        spike_array = pd.DataFrame(self.create_aligned_spikes_array().to_list())
        Z = spike_array.to_numpy()
        
        return hv.Raster(Z).opts(
            title=f"Cell {cell.cell_id} Aligned Spikes Heatmap",
            xlabel='Time', ylabel='Trial #', width=800, height=400,
            cmap='Cividis', spike_length=1, line_width=5, tools=['hover']
        )
    
    def get_PSTH_df(self, epok=[-500, 800], alignment_point='go_cue', smooth_ker_size=25, epok_buffer=0):
        assert isinstance(epok_buffer, int) and epok_buffer >= 0
        epok_start, epok_end = np.array(epok, dtype=int) + np.array([-epok_buffer, epok_buffer])
        # print(epok_start, epok_end)
        self.crop_aligned_spikes_to_epok(epok_start, epok_end, alignment_point)

        psth_df = pd.DataFrame(
            {'dir': d , 'PSTH_raw': np.array([]), 'PSTH': np.array([])} 
            for d in self.data['dir'].unique()
        )
        psth_df.set_index('dir', inplace=True)

        for dir, data in self.data.groupby('dir'):
            # agg_spikes = data['spikes_in_epok'].sum() / (data['spikes_in_epok'].sum().sum() * 0.001)
            agg_spikes = data['spikes_in_epok'].sum() / (data.shape[0] * 0.001)
            if smooth_ker_size:
                agg_spikes = gaussian_filter1d(agg_spikes, sigma=smooth_ker_size)
            psth_df.loc[dir, 'PSTH_raw'] = agg_spikes

        # avg_psth = psth_df['PSTH_raw'].sum() / (psth_df['PSTH_raw'].shape[0])
        psth_df['PSTH'] = psth_df['PSTH_raw'] #.apply(lambda x: x - avg_psth)

        # fix unfound directions
        # nans_arr = np.empty(avg_psth.shape)
        nans_arr = np.empty(psth_df['PSTH'][0].shape)
        nans_arr[:] = np.nan
        for dir in np.setdiff1d(np.arange(0, 360, step=45, dtype=int), self.directions):
            psth_df.loc[dir] = {'PSTH': nans_arr, 'PSTH_raw': nans_arr}

        time_frame = np.arange(epok_start, epok_end, 1, dtype=int)
        psth_df.attrs["Time"] = time_frame

        if epok_buffer > 0:
            # psth_df[['PSTH']] = psth_df.apply(
            #     lambda x: [a for a in np.array(x.to_list())[:, epok_buffer : -epok_buffer]],
            #     axis=1, result_type='expand'
            # )
            psth_df['PSTH_raw'] = psth_df.apply(
                lambda x: x['PSTH_raw'][epok_buffer : -epok_buffer], 
                axis=1
            ) 
            psth_df['PSTH'] = psth_df.apply(
                lambda x: x['PSTH'][epok_buffer : -epok_buffer], 
                axis=1
            )
            psth_df.attrs["Time"] = time_frame[epok_buffer : -epok_buffer]
             
        self.psth_df = psth_df
        return psth_df
    
    def get_PSTH_row(self, epok=[-500, 800], alignment_point='go_cue', smooth_ker_size=25, epok_buffer=0):
        psth_df = self.get_PSTH_df(
            epok=epok, alignment_point=alignment_point, smooth_ker_size=smooth_ker_size, epok_buffer=epok_buffer
        )
        return np.array(psth_df['PSTH'].to_list()).reshape(-1)
    
    def get_PSTH_rows_per_range(self, epok_len=50, step=1, epoks_range=[-50,50], alignment_point='go_cue', smooth_ker_size=25, epok_buffer=0):
        """
        Get a list of PSTH rows for the cell. PSTHs are of length epok_len and are aligned to alignment_point.
        The epos start at the ranges of epoks_range with a step of step and length of epok_len.
        """
        epok_start_times = np.arange(epoks_range[0], epoks_range[1], step)
        epok_end_times = epok_start_times + epok_len    
        epoks = np.vstack((epok_start_times, epok_end_times)).T
        print(epok_start_times, epok_end_times, epoks.shape, epoks[:3])
        with ProcessPoolExecutor() as executor:
            futures = [
                executor.submit(
                    self.get_PSTH_row, epok=epok, alignment_point=alignment_point, 
                    smooth_ker_size=smooth_ker_size, epok_buffer=epok_buffer
                ) 
                for epok in epoks
                ]
            psth_rows = [future.result() for future in futures]
        return psth_rows
    
    def plot_PSTH(self, epok=[-500, 800], alignment_point='go_cue', smooth_ker_size=25, epok_buffer=0):
        if self.psth_df is None:
            self.get_PSTH_df(
                epok=epok, alignment_point=alignment_point, 
                smooth_ker_size=smooth_ker_size, epok_buffer=epok_buffer
            )

        p = hv.Curve([])
        for dir in self.directions:
            p *= hv.Curve(
                np.vstack((self.psth_df.attrs["Time"], self.psth_df['PSTH'].loc[dir])).T ,
                label=f"{dir}˚"
            ).opts(
                color=self.DIR_COLORS[dir], muted_alpha=0, show_grid=True,
                title=f"Cell {self.cell_id} PSTH by dircetion", line_width=5,
                xlabel='Time', ylabel='Spikes / S', width=800, height=400
            )

        return p
        

analyzer = DataFrameAnalyzer(cell_df)
print("Unique cells:", analyzer.cells)
print("Unique sessions:", analyzer.sessions)
print("Unique trials:", analyzer.trials)
print("Number of cells:", len(analyzer.cells), "Number of sessions:", len(analyzer.sessions), "Number of trials:", len(analyzer.trials))
print("Monkeys:", analyzer.monkeys)
print("Is there a cell that has more than one session?", analyzer.has_multiple_sessions())

id = 9867
print(f"Sessions for cell {id}:", analyzer.get_sessions_by_cell(id))
print(f"trials for cell {id}:", analyzer.get_trials_by_cell(id))

cell = analyzer.get_cell_instance(id)
print(f"Cell instance for cell {id}:", cell)
print(f"Directions for cell {id}:", cell.directions)


Unique cells: [9867, 9868, 9869, 9870, 9871, 9872, 9876, 117, 121, 125, 129, 130, 131, 138, 140, 142, 122, 11, 12, 13, 15, 16, 17, 18, 21, 22, 23, 24, 29, 25, 26, 28, 9002, 842, 843, 845, 847, 848, 849, 850, 851, 852, 856, 857, 858, 861, 864, 865, 9454, 9455, 9457, 9458, 9459, 9461, 9465, 9466, 9468, 9463, 9456, 9138, 9139, 9142, 9143, 9148, 9151, 9147, 9140, 9150, 9040, 2253, 2256, 2259, 2269, 2271, 2272, 2277, 2278, 2280, 2281, 2284, 2287, 2291, 2292, 2294, 2296, 2297, 2301, 2302, 2303, 2304, 2305, 2312, 2316, 2317, 2318, 2323, 2328, 2329, 2333, 2334, 2335, 2336, 2250, 2252, 2254, 2255, 2257, 2260, 2261, 2262, 2263, 2264, 2265, 2266, 2267, 2268, 2274, 2275, 2279, 2282, 2283, 2285, 2286, 2288, 2289, 2290, 2293, 2295, 2298, 2300, 2306, 2307, 2308, 2309, 2311, 2313, 2314, 2315, 2319, 2320, 2322, 2325, 2330, 2258, 2273, 2310, 2326, 2299, 1738, 1739, 1740, 1741, 1742, 1744, 1745, 1746, 1749, 1754, 1761, 1762, 1764, 1765, 1766, 1768, 1770, 1771, 1772, 1773, 1774, 1775, 1776, 1780, 1788, 17

In [26]:
cell.data = cell.data[
    cell.data['type'].isin(['STOP']) & 
    cell.data['trial_failed'].isin([False]) &
    cell.data['ssd_number'].isin([2]) &
    cell.data['dir'].isin([180])
]

In [27]:
# spike_array = pd.DataFrame(cell.create_aligned_spikes_array().to_list())
# X = np.arange(spike_array.shape[1])
# Y = np.arange(spike_array.shape[0])
# Z = spike_array.to_numpy()
# hv.Raster(Z).opts(
#     title=f"Cell {cell.cell_id} Aligned Spikes Heatmap",
#     xlabel='Time', ylabel='Trial #', width=800, height=400,
#     cmap='Cividis'
# )

In [31]:
start, end = -200, 500
spikes_in_epok = cell.crop_aligned_spikes_to_epok(epok_start=start, epok_end=end, alignment_point='go_cue')
Z = pd.DataFrame(spikes_in_epok.to_list()).to_numpy()

# Use bounds to map column indices (0..Z.shape[1]) onto actual time range (-50..200)
time_axis = np.arange(start, end+1)
trial_axis = np.arange(Z.shape[0])

raster = hv.Image((time_axis, trial_axis, Z), kdims=['Time (ms)', 'Trial #'], vdims='Spikes').opts(
    title=f"Cell {cell.cell_id} Aligned Spikes Heatmap",
    xlabel='Time (ms)', ylabel='Trial #', width=800, height=400,
    cmap='Viridis', tools=['hover']
)
# raster = hv.Raster(Z).opts(
#     title=f"Cell {cell.cell_id} Aligned Spikes Heatmap",
#     xlabel='Time (ms)', ylabel='Trial #', width=800, height=400,
#     cmap='Viridis', tools=['hover']
# )
raster

:Image   [Time (ms),Trial #]   (Spikes)

In [29]:
cell.fix_spike_times_to_alignment()


272                       [[-156, 12]]
305       [[-631, -445, 45, 737, 763]]
317                      [[-703, 674]]
318                [[-824, -628, 791]]
382                [[-593, -264, 370]]
385                               [[]]
415                            [[809]]
422                [[-852, -188, 612]]
447                [[-435, -186, 256]]
452    [[-688, -439, -293, -159, -12]]
454                       [[160, 360]]
488            [[-220, 175, 336, 491]]
491                               [[]]
504      [[-694, -235, -88, -44, 142]]
508           [[-971, -797, 558, 736]]
538                       [[162, 473]]
Name: fixed_neural_data, dtype: object

In [30]:
cell.plot_cell_raster(alignment_point='go_cue', epok=[-50, 200])

ValueError: Unexpected option 'spike_length' for Raster type across all extensions. No similar options found.

In [ ]:
cell_df.apply(lambda row: len(row['neural_data']), axis=1).value_counts()

In [ ]:
def get_spikes_data(row, epok=None):
    if epok is not None:
            return np.argwhere(row['spikes_in_epok'] != 0).flatten() + epok[0]
    else:
        
        return row['fixed_neural_data'].astype(int) if not len(row['fixed_neural_data']) == 0 else np.array([-1], dtype=int)
        

cell.fix_spike_times_to_alignment()
# cell.data
overlay = hv.NdOverlay(
    {
        i: hv.Spikes(
            get_spikes_data(cell.data.iloc[i]), kdims='Time'
        ).opts(
            position=i, color=cell.DIR_COLORS[cell.data.iloc[i]['dir']]
        ) 
        for i in range(64, 65) #(cell.data.shape[0])
    }
).opts(
    # yticks=[i for i in np.arange(self.data.shape[0], step=10)],
    ylabel='Trial #', show_legend=True,
    title=f"Cell {cell.cell_id} Raster Plot aligned to "
)
overlay

In [ ]:
cell.data.iloc[64]['fixed_neural_data']

hv.Spikes(
    [-1], kdims='Time'
)

In [ ]:
cell.data.iloc[0]['fixed_neural_data']